## Apex Wealth Data Pipeline

In [1]:
# Import necessary libraries
import requests
import pandas as pd
import psycopg2 as pg
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os


In [2]:
# Load environment variables from .env file
load_dotenv()

# Get API key from environment variable
api_key = os.getenv('API_KEY')

## Extraction

In [103]:
# define url endpoint
symbols = ['AAPL', 'MSFT', 'GOOGL', 'AMZN']

def extract_data(symbols):
    all_records = []
    for symbol in symbols:
        url = f'https://api.twelvedata.com/time_series?symbol={symbol}&interval=1min&apikey={api_key}'
        response = requests.get(url)
        if response.status_code == 200:
            print("API request successful.")
        data = response.json()

        for record in data["values"]:
            record['symbol'] = symbol

        all_records.extend(data["values"])
    return all_records

data = extract_data(symbols)

API request successful.
API request successful.
API request successful.
API request successful.


In [104]:
data

[{'datetime': '2026-06-22 15:59:00',
  'open': '296.95001',
  'high': '297.29001',
  'low': '296.79001',
  'close': '296.79001',
  'volume': '1046896',
  'symbol': 'AAPL'},
 {'datetime': '2026-06-22 15:58:00',
  'open': '297.059998',
  'high': '297.14001',
  'low': '296.82001',
  'close': '296.94000',
  'volume': '473949',
  'symbol': 'AAPL'},
 {'datetime': '2026-06-22 15:57:00',
  'open': '297.049988',
  'high': '297.21',
  'low': '297.019989',
  'close': '297.059998',
  'volume': '303884',
  'symbol': 'AAPL'},
 {'datetime': '2026-06-22 15:56:00',
  'open': '297.25500',
  'high': '297.26999',
  'low': '297.019989',
  'close': '297.040009',
  'volume': '255354',
  'symbol': 'AAPL'},
 {'datetime': '2026-06-22 15:55:00',
  'open': '297.57501',
  'high': '297.67999',
  'low': '297.2',
  'close': '297.26001',
  'volume': '280455',
  'symbol': 'AAPL'},
 {'datetime': '2026-06-22 15:54:00',
  'open': '297.22000',
  'high': '297.61',
  'low': '296.91000',
  'close': '297.55499',
  'volume': '2

In [77]:
# get the 'high' price from the second entry in the 'values' list
data['values'][1]['high']

'297.14001'

In [123]:
print(symbols)

['AAPL', 'MSFT', 'GOOGL', 'AMZN']


## Transformation

In [120]:
def transform_data(data):

    
    df = pd.DataFrame(data)

    df['datetime'] = pd.to_datetime(df['datetime'])

    df = df.astype({
        'open': float,
        'high': float,
        'low': float,
        'close': float,
        'volume': int
    })

    return df

In [134]:
df = df[
    ['symbol', 'datetime', 'open', 'high', 'low', 'close', 'volume']
]

In [135]:
df

,symbol,datetime,open,high,low,close,volume
0,AAPL,2026-06-22 15:59:00,296.950010,297.29001,296.790010,296.790010,1046896
1,AAPL,2026-06-22 15:58:00,297.059998,297.14001,296.820010,296.940000,473949
2,AAPL,2026-06-22 15:57:00,297.049988,297.21000,297.019989,297.059998,303884
3,AAPL,2026-06-22 15:56:00,297.255000,297.26999,297.019989,297.040009,255354
4,AAPL,2026-06-22 15:55:00,297.575010,297.67999,297.200000,297.260010,280455
...,...,...,...,...,...,...,...
115,AMZN,2026-06-22 15:34:00,233.320010,233.32001,233.000000,233.039993,194663
116,AMZN,2026-06-22 15:33:00,233.554990,233.56000,233.289990,233.320010,186833
117,AMZN,2026-06-22 15:32:00,233.735000,233.73500,233.539990,233.575000,228126
118,AMZN,2026-06-22 15:31:00,233.804990,233.83000,233.650100,233.740010,169914


## Loading

In [140]:
load_dotenv()
API_KEY = os.getenv('API_KEY')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

In [141]:
DB_NAME

'apex_db_eu'

In [144]:
# create connection url
db_url = f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'

# create sqlalchemy engine
engine = create_engine(db_url)

# load dataframe to sql table
df.to_sql('stock_multi_prices', engine, if_exists='append', index=False)

print("Data has been successfully loaded to the database.")

Data has been successfully loaded to the database.


In [145]:
# save as csv file
df.to_csv('stock_prices.csv', index=False)
print("Data has been successfully saved to CSV file.")  

Data has been successfully saved to CSV file.
